In [37]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [38]:
csv_path = '../data/dataset/pull_request_data_processed.csv'
data = pd.read_csv(csv_path)

In [39]:
data.head(1)

,state_open,has_title,has_body,is_locked,is_closed,has_assignees,has_labels,author_association,user_type,repo_size,repo_stargazer_count,repo_watcher_count,repo_has_issue,repo_has_projects,repo_has_downloads,repo_has_wiki,repo_has_pages,repo_has_discussions,repo_fork_count,merged
0,1,1,1,0,0,0,1,0,0,1335380,111257,111257,1,1,1,0,0,0,31553,0


In [40]:
### checks initial initial class distribution
data['merged'].value_counts()

merged
1    19458
0    10542
Name: count, dtype: int64

In [41]:

X = data.iloc[:,:-1].values
y = data['merged'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.30,
                                                    random_state=15, stratify=y)

In [42]:

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3,5,7,10]
}

scoring = 'f1'

clf = GridSearchCV(DecisionTreeClassifier(), param_grid=param_grid, scoring=scoring, cv=10)

In [43]:
clf.fit(X_train, y_train)


print("Mejor combinación de parámetros:")
print(clf.best_params_)
 
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

Mejor combinación de parámetros:
{'criterion': 'gini', 'max_depth': 10}
              precision    recall  f1-score   support

           0       0.94      0.66      0.78      3163
           1       0.84      0.98      0.90      5837

    accuracy                           0.87      9000
   macro avg       0.89      0.82      0.84      9000
weighted avg       0.88      0.87      0.86      9000



Mejor combinación de parámetros:
{'criterion': 'entropy', 'max_depth': 10}
              precision    recall  f1-score   support

           0       0.91      0.78      0.84       820
           1       0.87      0.95      0.91      1340

    accuracy                           0.89      2160
   macro avg       0.89      0.87      0.88      2160
weighted avg       0.89      0.89      0.88      2160

# DOING SOME BALANCE

In [ ]:
csv_path = '../data/dataset/pull_request_data_processed.csv'
data = pd.read_csv(csv_path)

In [45]:
data['merged'].value_counts()

merged
1    19458
0    10542
Name: count, dtype: int64

In [54]:
X = data.drop(columns=['merged'])
y = data['merged']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.30, random_state=15, stratify=y)


### UNDERSAMPLE

In [50]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X_train, y_train)

print("Balanced class distribution (undersampling):")
print(y_resampled.value_counts())

Balanced class distribution (undersampling):
merged
0    7379
1    7379
Name: count, dtype: int64


In [51]:
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3,5,7,10]
}

scoring = 'f1'

clf = GridSearchCV(DecisionTreeClassifier(), param_grid=param_grid, scoring=scoring, cv=10)

In [52]:
clf.fit(X_resampled, y_resampled)


print("Mejor combinación de parámetros:")
print(clf.best_params_)
 
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

Mejor combinación de parámetros:
{'criterion': 'gini', 'max_depth': 10}
              precision    recall  f1-score   support

           0       0.85      0.71      0.78      3163
           1       0.86      0.93      0.89      5837

    accuracy                           0.86      9000
   macro avg       0.85      0.82      0.83      9000
weighted avg       0.85      0.86      0.85      9000



### OVERSAMPLE

In [55]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train, y_train)

print("Balanced class distribution (oversampling):")
print(y_resampled.value_counts())

Balanced class distribution (oversampling):
merged
1    13621
0    13621
Name: count, dtype: int64


In [56]:
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3,5,7,10]
}

scoring = 'f1'

clf = GridSearchCV(DecisionTreeClassifier(), param_grid=param_grid, scoring=scoring, cv=10)

In [57]:
clf.fit(X_resampled, y_resampled)


print("Mejor combinación de parámetros:")
print(clf.best_params_)
 
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

Mejor combinación de parámetros:
{'criterion': 'entropy', 'max_depth': 10}
              precision    recall  f1-score   support

           0       0.85      0.72      0.78      3163
           1       0.86      0.93      0.89      5837

    accuracy                           0.85      9000
   macro avg       0.85      0.82      0.83      9000
weighted avg       0.85      0.85      0.85      9000

